In [6]:
import polars as pl

splits = {'train': 'main/train-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}
df_main_train = pl.read_parquet('hf://datasets/openai/gsm8k/' + splits['train'])
df_main_test = pl.read_parquet('hf://datasets/openai/gsm8k/' + splits['test'])


splits = {'train': 'socratic/train-00000-of-00001.parquet', 'test': 'socratic/test-00000-of-00001.parquet'}
df_socratic = pl.read_parquet('hf://datasets/openai/gsm8k/' + splits['train'])

In [13]:
print(df_main_train.shape)
print(df_main_test.shape)

df_main_train.write_parquet('raw/gsm8k_train.parquet')
df_main_test.write_parquet('raw/gsm8k_test.parquet')

(7473, 2)
(1319, 2)


In [3]:
math1_df = pl.read_parquet("data/generated/math1_results_long.parquet")
math2_df = pl.read_parquet("data/generated/math2_results_long.parquet")
aime_df = pl.read_parquet("data/generated/aime_results_long.parquet")

In [4]:
print(math1_df.shape)
print(math2_df.shape)
print(aime_df.shape)
math1_df.head()

(125000, 9)
(125000, 9)
(18660, 9)


question_id,prompt,solution_col,generated_think_text,generated_text,target_think_tokens,generated_think_tokens,latency_sec,is_correct
i64,str,str,str,str,i64,i64,f64,bool
0,"""Let \[f(x) = \left\{ \begin{ar…","""For the piecewise function to …","""Okay, set limits at x=2 and x=…","""Let \[f(x) = \left\{ \begin{ar…",100,22,0.928551,false
0,"""Let \[f(x) = \left\{ \begin{ar…","""For the piecewise function to …","""Okay, for continuity at x=2 an…","""Let \[f(x) = \left\{ \begin{ar…",357,90,0.928551,false
0,"""Let \[f(x) = \left\{ \begin{ar…","""For the piecewise function to …","""Okay, to ensure continuity, th…","""Let \[f(x) = \left\{ \begin{ar…",615,182,0.928551,true
0,"""Let \[f(x) = \left\{ \begin{ar…","""For the piecewise function to …","""Okay, so I need to find a and …","""Let \[f(x) = \left\{ \begin{ar…",873,283,0.928551,true
0,"""Let \[f(x) = \left\{ \begin{ar…","""For the piecewise function to …","""Okay, so I need to find a and …","""Let \[f(x) = \left\{ \begin{ar…",1131,460,0.928551,true


In [5]:
aime_df.head()

question_id,prompt,solution_col,generated_think_text,generated_text,target_think_tokens,generated_think_tokens,latency_sec,is_correct
i64,str,str,str,str,i64,i64,f64,bool
0,"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, let me try. Use change o…","""Let $x$ , $y$ and $z$ all exce…",100,28,1.130943,false
0,"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, using change of base for…","""Let $x$ , $y$ and $z$ all exce…",357,86,1.130943,false
0,"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, so I have log base x of …","""Let $x$ , $y$ and $z$ all exce…",615,237,1.130943,false
0,"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, so I need to find log ba…","""Let $x$ , $y$ and $z$ all exce…",873,363,1.130943,false
0,"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, so I need to find log ba…","""Let $x$ , $y$ and $z$ all exce…",1131,599,1.130943,false


In [13]:
#Next to each question id, append aime-id
aime_df_new = aime_df.with_columns(
    pl.col("question_id").map_elements(lambda x: f"aime_{x}").alias("aime_question_id")
)

In [14]:
#move question_id to first column
aime_df_new = aime_df_new.select(["question_id"] + [col for col in aime_df_new.columns if col != "question_id"])

In [15]:
aime_df_new.head()

question_id,prompt,solution_col,generated_think_text,generated_text,target_think_tokens,generated_think_tokens,latency_sec,is_correct,aime_question_id
i64,str,str,str,str,i64,i64,f64,bool,str
0,"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, let me try. Use change o…","""Let $x$ , $y$ and $z$ all exce…",100,28,1.130943,false,"""aime_0"""
0,"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, using change of base for…","""Let $x$ , $y$ and $z$ all exce…",357,86,1.130943,false,"""aime_0"""
0,"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, so I have log base x of …","""Let $x$ , $y$ and $z$ all exce…",615,237,1.130943,false,"""aime_0"""
0,"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, so I need to find log ba…","""Let $x$ , $y$ and $z$ all exce…",873,363,1.130943,false,"""aime_0"""
0,"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, so I need to find log ba…","""Let $x$ , $y$ and $z$ all exce…",1131,599,1.130943,false,"""aime_0"""


In [12]:
aime_df_new.write_parquet("aime_results_long_with_aime_id.parquet")

In [88]:
from math_equivalence import is_equiv 
import re
def extract_boxed(s: str) -> str | None:
    if not s: return None
    m = re.search(r"\\boxed\{([^}]*)\}", s)
    return m.group(1).strip() if m else None

def extract_think_text(full_text: str) -> str:
    match = re.search(r"<think>(.*?)</think>", full_text, flags=re.DOTALL)
    return match.group(1).strip() if match else ""

def evaluate_answer(expected_answer: str, generated_answer: str) -> bool:
    exp_val = extract_boxed(expected_answer)
    gen_val = extract_boxed(generated_answer)

    if exp_val is None:
        exp_val = expected_answer.strip()
    print(f"Generated answer: {gen_val}, Expected answer: {exp_val}")

    if exp_val is None or gen_val is None:
        return False
    
    return is_equiv(gen_val, exp_val)


In [25]:
#drop column question_id and rename aime_question_id to question_id
aime_df_new = aime_df_new.drop("question_id").rename({"aime_question_id": "question_id"})

In [28]:
#accuracy, percentage of quesiton with is_correct = True
accuracy = aime_df_new.filter(pl.col("is_correct") == True).height / aime_df_new.height
print(f"Accuracy: {accuracy:.2%}")
aime_df_new.head()

Accuracy: 42.80%


prompt,solution_col,generated_think_text,generated_text,target_think_tokens,generated_think_tokens,latency_sec,is_correct,question_id
str,str,str,str,i64,i64,f64,bool,str
"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, let me try. Use change o…","""Let $x$ , $y$ and $z$ all exce…",100,28,1.130943,true,"""aime_0"""
"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, using change of base for…","""Let $x$ , $y$ and $z$ all exce…",357,86,1.130943,true,"""aime_0"""
"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, so I have log base x of …","""Let $x$ , $y$ and $z$ all exce…",615,237,1.130943,true,"""aime_0"""
"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, so I need to find log ba…","""Let $x$ , $y$ and $z$ all exce…",873,363,1.130943,true,"""aime_0"""
"""Let $x$ , $y$ and $z$ all exce…","""60""","""Okay, so I need to find log ba…","""Let $x$ , $y$ and $z$ all exce…",1131,599,1.130943,false,"""aime_0"""


In [46]:
aime_df.shape

(18660, 9)

In [62]:
#Sample some incorrect answers and show only solution_col and generated_text filtered to the boxed element
incorrect_answers = aime_df_new.filter(pl.col("is_correct") == False)
incorrect_answers = incorrect_answers.select(["solution_col", "generated_text"])

In [85]:
short_csv = incorrect_answers.sample(10)
short_csv["solution_col"].unique()

solution_col
str
"""90"""
"""183"""
"""418"""
"""429"""
"""817"""
"""484"""
"""32"""
"""400"""
"""65"""


In [89]:
#apply evaluate_answer to all rows in short_csv and show is_correct column
short_csv = short_csv.with_columns(
    pl.struct(["solution_col", "generated_text"]).map_elements(
        lambda row: evaluate_answer(row["solution_col"], row["generated_text"])
    ).alias("is_correct")
)   
short_csv.head(20)

Generated answer: None, Expected answer: 
Generated answer: None, Expected answer: 
Generated answer: 259, Expected answer: 400
Generated answer: 31, Expected answer: 817
Generated answer: 3, Expected answer: 65
Generated answer: 2009, Expected answer: 90
Generated answer: 568, Expected answer: 32
Generated answer: 495, Expected answer: 947
Generated answer: \dfrac{143, Expected answer: 429
Generated answer: 192, Expected answer: 418
Generated answer: 274, Expected answer: 484
Generated answer: 15, Expected answer: 183


solution_col,generated_text,is_correct
str,str,bool
"""400""","""Let $f(n)$ be the integer clos…",false
"""817""","""A circle of radius 1 is random…",false
"""65""","""In triangle $ABC$ , $AB=\sqrt{…",false
"""90""","""The terms of the sequence $\{a…",false
"""32""","""Let $S$ be the increasing sequ…",false
"""947""","""A sample of 121 integers is gi…",false
"""429""","""A certain function $f$ has the…",false
"""418""","""Let $P(x)$ be a polynomial wit…",false
"""484""","""Let $ABCDE$ be a convex pentag…",false
